[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

# Lesson 13 — Comprehensions and Functional Tools

**Module 1 — Python Fundamentals** | ⏱ 25 min

Python offers a suite of elegant tools for transforming and filtering data without writing explicit loops. Comprehensions are a concise, Pythonic way to create new collections from existing ones. Generator expressions produce values lazily — one at a time — making them memory-efficient for large datasets. The functional tools `map()`, `filter()`, and `reduce()` apply functions to sequences, enabling a pipeline-style approach to data transformation.

## Learning Objectives
- Write list, dict, and set comprehensions with conditional filtering
- Use nested comprehensions to process 2D data
- Understand generator expressions and lazy evaluation
- Compare memory usage between lists and generators
- Use `map()`, `filter()`, and `functools.reduce()`
- Build a practical data transformation pipeline using comprehensions

## List Comprehensions

A list comprehension creates a new list by applying an expression to each item in an iterable, optionally filtering items with a condition. The syntax is `[expression for item in iterable if condition]`. Comprehensions are typically faster than equivalent `for` loops because they are optimised by the Python interpreter. They are most readable when the logic fits on one line — for anything more complex, a regular loop is clearer.

In [ ]:
# Equivalent approaches — loop vs comprehension
numbers = range(1, 21)

# Traditional loop
squares_loop = []
for n in numbers:
    if n % 2 == 0:
        squares_loop.append(n ** 2)

# List comprehension — same result, more concise
squares_comp = [n ** 2 for n in numbers if n % 2 == 0]

print("Even squares (loop):        ", squares_loop)
print("Even squares (comprehension):", squares_comp)
print("Results match:", squares_loop == squares_comp)

# Comprehensions with string operations
csv_header = "  Name , Age , Email , Department  "
columns = [col.strip().lower().replace(" ", "_") for col in csv_header.split(",")]
print(f"\nNormalised columns: {columns}")

# Comprehension with function call
import math
circle_areas = [round(math.pi * r**2, 2) for r in range(1, 8)]
print(f"Circle areas (r=1..7): {circle_areas}")

In [ ]:
# Practical: parse and clean a raw dataset
raw_data = [
    "  alice,28,engineer,95000  ",
    "BOB,35,MANAGER,120000",
    "carol,,developer,88000",    # Missing age
    "DAVID,42,director,150000",
    "eve,29,engineer,-500",       # Invalid salary
    "frank,31,analyst,72000",
]

def parse_record(line):
    """Parse a CSV line into a dict. Returns None if the record is invalid."""
    parts = line.strip().split(",")
    if len(parts) != 4:
        return None
    name, age_str, role, salary_str = parts
    try:
        age = int(age_str)
        salary = int(salary_str)
    except ValueError:
        return None
    if salary <= 0 or age <= 0:
        return None
    return {"name": name.strip().title(), "age": age, "role": role.strip().lower(), "salary": salary}

# Filter and transform in one comprehension
records = [r for line in raw_data if (r := parse_record(line)) is not None]
print(f"Valid records: {len(records)} out of {len(raw_data)}")
for r in records:
    print(f"  {r['name']:<10} | {r['role']:<12} | ${r['salary']:>8,}")

## Dict and Set Comprehensions

The comprehension syntax extends to dictionaries and sets. A **dict comprehension** creates a dictionary: `{key_expr: value_expr for item in iterable if condition}`. A **set comprehension** creates a set: `{expr for item in iterable if condition}`. These follow the same rules as list comprehensions — they are readable for simple transformations and should be replaced with explicit loops for complex logic.

In [ ]:
# Dict comprehension examples

# 1. Invert a dictionary (swap keys and values)
country_code = {"CA": "Canada", "US": "United States", "GB": "United Kingdom", "AU": "Australia"}
code_lookup = {name: code for code, name in country_code.items()}
print(f"Lookup 'Canada': {code_lookup['Canada']}")

# 2. Transform values
prices_usd = {"apple": 1.20, "banana": 0.75, "cherry": 4.50, "grape": 3.20}
exchange_rate = 1.38  # USD to CAD
prices_cad = {item: round(price * exchange_rate, 2) for item, price in prices_usd.items()}
print(f"Prices in CAD: {prices_cad}")

# 3. Filter and transform: only expensive items
premium_items = {item: price for item, price in prices_usd.items() if price > 2.00}
print(f"Premium items (> $2.00 USD): {premium_items}")

# Set comprehension: unique values from a messy list
log_levels = ["INFO", "warning", "ERROR", "info", "WARNING", "error", "DEBUG"]
unique_levels = {level.upper() for level in log_levels}
print(f"\nUnique log levels: {sorted(unique_levels)}")

# Vowel set from a sentence
sentence = "The quick brown fox"
vowels_used = {char.lower() for char in sentence if char.lower() in 'aeiou'}
print(f"Vowels used: {sorted(vowels_used)}")

## Nested Comprehensions

Comprehensions can be nested — an outer loop and an inner loop in a single expression. This is equivalent to two nested `for` loops. Nested comprehensions are powerful for working with matrices, grids, or combinations, but they can quickly become unreadable. A good rule: if the comprehension requires more than 2 loops or complex conditions, use a regular loop or break it into steps.

In [ ]:
# Nested comprehension — flatten a 2D matrix into a 1D list
matrix = [
    [1,  2,  3,  4],
    [5,  6,  7,  8],
    [9, 10, 11, 12]
]

# Note the order: first loop is outermost, second loop is innermost
flat = [val for row in matrix for val in row]
print(f"Flattened: {flat}")

# Transpose a matrix (rows become columns)
transposed = [[row[i] for row in matrix] for i in range(len(matrix[0]))]
print("\nOriginal matrix:")
for row in matrix:
    print(f"  {row}")
print("Transposed:")
for row in transposed:
    print(f"  {row}")

# Generate all (row, col) pairs where value is above threshold
threshold = 7
high_values = [(r, c, matrix[r][c]) for r in range(len(matrix))
                                      for c in range(len(matrix[r]))
                                      if matrix[r][c] > threshold]
print(f"\nValues > {threshold}: {high_values}")

## Generator Expressions

A **generator expression** has the same syntax as a list comprehension but uses parentheses `()` instead of brackets `[]`. The key difference is **lazy evaluation**: a generator produces values one at a time on demand, rather than computing all values upfront and storing them in memory. This makes generators ideal for large datasets or infinite sequences where materialising the entire list would be impractical or impossible.

In [ ]:
import sys

# Memory comparison: list vs generator
n = 1_000_000

# List comprehension — stores ALL values in memory
list_comp = [x ** 2 for x in range(n)]
list_size = sys.getsizeof(list_comp)
print(f"List of {n:,} squares:      {list_size:>12,} bytes")

# Generator expression — stores almost nothing
gen_exp = (x ** 2 for x in range(n))
gen_size = sys.getsizeof(gen_exp)
print(f"Generator for {n:,} squares: {gen_size:>12,} bytes")
print(f"Memory ratio: {list_size / gen_size:,.0f}x larger for the list")

# Generators are lazy — values computed on demand
def number_generator(start, end):
    """A generator function — yields values one at a time."""
    current = start
    while current <= end:
        yield current   # Pause here and return the value; resume on next()
        current += 1

gen = number_generator(1, 5)
print(f"\nGenerator object: {gen}")
print(next(gen))  # 1
print(next(gen))  # 2
print(list(gen))  # [3, 4, 5] — consume remaining values

In [ ]:
# Practical generator use: processing a large log file line by line
# (We simulate with a list, but the pattern works for real files too)

def generate_log_lines(lines):
    """Yield log lines that are errors — simulates reading a large file."""
    for line in lines:
        if "ERROR" in line or "CRITICAL" in line:
            yield line.strip()

# Simulate log data (in a real scenario this would be a huge file)
log_data = [
    "2024-01-15 10:00 INFO  App started",
    "2024-01-15 10:01 INFO  Connected to database",
    "2024-01-15 10:05 ERROR Database query timeout",
    "2024-01-15 10:06 INFO  Retrying connection",
    "2024-01-15 10:07 ERROR Max retries exceeded",
    "2024-01-15 10:08 CRITICAL Service unavailable",
    "2024-01-15 10:09 INFO  Failover initiated",
]

# The generator processes one line at a time — no huge list in memory
error_gen = generate_log_lines(log_data)
print("Error/Critical log entries:")
for line in error_gen:
    print(f"  >> {line}")

# Generator expressions work as arguments to sum(), max(), any(), all()
prices = [10.99, 25.50, 5.00, 15.75, 8.25]

# sum() with a generator — no intermediate list created
total = sum(price * 1.13 for price in prices)  # Add 13% tax
print(f"\nTotal with tax: ${total:.2f}")

# any() and all() short-circuit with generators — very efficient
have_expensive = any(p > 20 for p in prices)  # Stops at first True
all_positive   = all(p > 0 for p in prices)   # Stops at first False
print(f"Any > $20: {have_expensive}, All positive: {all_positive}")

## map(), filter(), and reduce()

`map()` applies a function to every item in an iterable. `filter()` keeps only items for which a function returns `True`. Both return lazy iterators (like generators). `reduce()` from `functools` cumulatively applies a function to reduce a sequence to a single value. In modern Python, comprehensions are usually preferred over `map()`/`filter()` for readability, but these functional tools are still widely used.

In [ ]:
from functools import reduce

# map() — apply a function to every element
temperatures_c = [0, 20, 37, 100, -40]

# Using map with a lambda
temperatures_f = list(map(lambda c: c * 9/5 + 32, temperatures_c))
print(f"Celsius:    {temperatures_c}")
print(f"Fahrenheit: {temperatures_f}")

# Using map with a named function (cleaner for complex transformations)
def celsius_to_fahrenheit(c):
    return round(c * 9/5 + 32, 1)

converted = list(map(celsius_to_fahrenheit, temperatures_c))
print(f"Fahrenheit: {converted}")

# filter() — keep only elements where function returns True
numbers = [-5, -3, -1, 0, 2, 4, 6, 8, 11, 13]
positives = list(filter(lambda n: n > 0, numbers))
evens     = list(filter(lambda n: n % 2 == 0, numbers))
print(f"\nPositives: {positives}")
print(f"Evens:     {evens}")

# Equivalent comprehensions (usually preferred)
positives_comp = [n for n in numbers if n > 0]
print(f"Positives (comp): {positives_comp}")

In [ ]:
from functools import reduce

# reduce() — cumulatively combine elements to get a single result
numbers = [1, 2, 3, 4, 5]

# Manual equivalent: ((((1+2)+3)+4)+5) = 15
total = reduce(lambda acc, x: acc + x, numbers)
print(f"Sum via reduce: {total}")

# Product of all numbers
product = reduce(lambda acc, x: acc * x, numbers)
print(f"Product via reduce: {product}")

# Find maximum without using max() — educational example
maximum = reduce(lambda a, b: a if a > b else b, numbers)
print(f"Maximum via reduce: {maximum}")

# Practical: combine log entries into one string using reduce
log_messages = ["Connected", "Authenticated", "Queried DB", "Response sent"]
full_log = reduce(lambda a, b: a + " -> " + b, log_messages)
print(f"\nLog chain: {full_log}")

# Chaining map, filter, reduce into a pipeline
raw_salaries = ["$95,000", "$72,000", "$120,000", "$invalid", "$88,000"]

# Step 1: parse (map)
def parse_salary(s):
    try:
        return int(s.replace("$", "").replace(",", ""))
    except ValueError:
        return None

# Step 2: filter out None
# Step 3: sum the valid ones
valid_salaries = list(filter(None, map(parse_salary, raw_salaries)))
total = reduce(lambda a, b: a + b, valid_salaries)
average = total / len(valid_salaries)
print(f"\nValid salaries: {valid_salaries}")
print(f"Total payroll: ${total:,}")
print(f"Average salary: ${average:,.0f}")

## Real-World Pipeline: Parsing CSV-like Data

Let us combine everything in this lesson into a realistic data transformation pipeline. We have raw CSV-like data from a sales report and need to clean, filter, aggregate, and format it — all using comprehensions and functional tools.

In [ ]:
from functools import reduce
from collections import defaultdict

# Raw sales data: date, region, product, quantity, unit_price
raw_sales = """
2024-01,North,Laptop,3,999.99
2024-01,South,Mouse,15,29.99
2024-01,North,Keyboard,8,79.99
2024-02,East,Laptop,5,999.99
2024-02,South,Monitor,2,349.99
2024-02,North,Mouse,20,29.99
2024-01,East,Keyboard,12,79.99
2024-02,East,Mouse,10,29.99
bad-data,row,here
2024-01,West,Monitor,3,349.99
""".strip().split("\n")

# Step 1: Parse and validate — list comprehension with walrus operator
def parse_sale(line):
    parts = line.split(",")
    if len(parts) != 5:
        return None
    try:
        return {
            "month":    parts[0],
            "region":   parts[1],
            "product":  parts[2],
            "qty":      int(parts[3]),
            "price":    float(parts[4]),
            "revenue":  int(parts[3]) * float(parts[4])
        }
    except ValueError:
        return None

sales = [s for line in raw_sales if (s := parse_sale(line)) is not None]
print(f"Parsed {len(sales)} valid records from {len(raw_sales)} lines")

# Step 2: Filter — only sales with revenue > $500
big_sales = [s for s in sales if s["revenue"] > 500]
print(f"High-value sales (> $500): {len(big_sales)}")

# Step 3: Aggregate revenue by region using defaultdict
revenue_by_region = defaultdict(float)
for s in sales:
    revenue_by_region[s["region"]] += s["revenue"]

# Dict comprehension to round values
revenue_by_region = {region: round(rev, 2) for region, rev in revenue_by_region.items()}

# Step 4: Sort and display
print("\nRevenue by Region:")
for region, rev in sorted(revenue_by_region.items(), key=lambda x: x[1], reverse=True):
    bar = "#" * int(rev // 500)
    print(f"  {region:<6}: ${rev:>9,.2f}  {bar}")

# Total using reduce
total_revenue = reduce(lambda acc, s: acc + s["revenue"], sales, 0)
print(f"\nTotal revenue: ${total_revenue:,.2f}")

# Best-selling products by quantity — dict comprehension with sum
from collections import defaultdict
qty_by_product = defaultdict(int)
for s in sales:
    qty_by_product[s["product"]] += s["qty"]
print("\nUnits sold by product:")
for prod, qty in sorted(qty_by_product.items(), key=lambda x: x[1], reverse=True):
    print(f"  {prod:<10}: {qty} units")

## Practice Exercises

1. Write a single list comprehension that generates all Pythagorean triples `(a, b, c)` where `a <= b <= c` and all values are less than 50. A Pythagorean triple satisfies `a² + b² == c²`.
2. Given a list of sentences, use a generator expression to yield only sentences that contain more than 5 words AND have at least one word longer than 8 characters. Process the generator lazily without materialising the full list.
3. Use `map()` and `filter()` (no comprehensions) to process a list of raw temperature strings like `["22.5C", "invalid", "31.0F", "18C", "bad"]` — parse them, convert Fahrenheit to Celsius, filter out invalid entries, and round to 1 decimal place.
4. Build a pipeline using `reduce()` to compute a running total — given a list of transaction amounts (positive for deposits, negative for withdrawals), produce a list of running balances at each step.